# LedgerShield Exquisite Training Colab

This notebook reruns the additive LedgerShield Exquisite training flow in Colab: self-play candidate generation, environment-in-the-loop GRPO, fast-profile 1.5B SFT scaling, optional DPO falsifier distillation, and the final evaluation/plot/dashboard/report rebuild.

The original benchmark SFT proof remains documented separately in `training/LedgerShield_OpenEnv_TRL_Training_Colab.ipynb`. This notebook is only for the additive `training/exquisite/` pipeline.

## Runtime Notes

- Use a GPU runtime for `fast_gpu` or `full_reference`.
- `fast_gpu` is the recommended judge rerun profile.
- `full_reference` is closer to the completed artifact story but will take much longer.
- Do not paste private Hugging Face tokens into committed cells. If needed, set them in Colab secrets or runtime environment variables.

In [ ]:
from pathlib import Path

if not Path('README.md').exists():
    !git clone https://github.com/BiradarScripts/Meta-s-LedgerShield.git
    %cd Meta-s-LedgerShield
else:
    print('Using existing checkout')


In [ ]:
!pip install -q -r training/requirements-training.txt
!nvidia-smi || true


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess

RUN_PROFILE = 'fast_gpu'  # one of: fast_gpu, full_reference
RUN_DPO = True

PROFILE_CONFIGS = {
    'fast_gpu': {
        'description': 'Judge-friendly rerun profile for a Colab GPU runtime.',
        'selfplay': {
            'case_limit': 24,
            'eval_case_limit': 6,
            'num_generations': 4,
            'max_new_tokens': 1200,
            'temperature': 0.7,
            'top_p': 0.95,
            'mode': 'model',
        },
        'grpo': {
            'case_limit': 24,
            'eval_case_limit': 6,
            'max_steps': 60,
            'batch_size': 1,
            'gradient_accumulation_steps': 4,
            'num_generations': 4,
            'max_completion_length': 1200,
            'reward_eval_case_limit': 2,
            'reward_eval_max_new_tokens': 1200,
        },
        'sft_1_5b': {
            'case_limit': 24,
            'max_steps': 300,
            'max_new_tokens': 1200,
            'model_eval_case_limit': 3,
            'skip_base_model_eval': True,
            'reward_eval_interval': 0,
            'no_plots': True,
        },
        'dpo': {
            'case_limit': 12,
            'eval_case_limit': 6,
            'max_steps': 60,
            'batch_size': 1,
            'gradient_accumulation_steps': 4,
            'eval_max_new_tokens': 1200,
        },
    },
    'full_reference': {
        'description': 'Closer to the completed repository run profile and therefore slower and more expensive.',
        'selfplay': {
            'case_limit': 45,
            'eval_case_limit': 9,
            'num_generations': 8,
            'max_new_tokens': 2200,
            'temperature': 0.8,
            'top_p': 0.95,
            'mode': 'model',
        },
        'grpo': {
            'case_limit': 36,
            'eval_case_limit': 9,
            'max_steps': 250,
            'batch_size': 1,
            'gradient_accumulation_steps': 4,
            'num_generations': 8,
            'max_completion_length': 2200,
            'reward_eval_case_limit': 3,
            'reward_eval_max_new_tokens': 2200,
        },
        'sft_1_5b': {
            'case_limit': 45,
            'max_steps': 300,
            'max_new_tokens': 1200,
            'model_eval_case_limit': 3,
            'skip_base_model_eval': True,
            'reward_eval_interval': 0,
            'no_plots': True,
        },
        'dpo': {
            'case_limit': 18,
            'eval_case_limit': 9,
            'max_steps': 180,
            'batch_size': 1,
            'gradient_accumulation_steps': 4,
            'eval_max_new_tokens': 1800,
        },
    },
}

cfg = PROFILE_CONFIGS[RUN_PROFILE]
ARTIFACT_ROOT = Path(f'artifacts/exquisite-training-colab-{RUN_PROFILE}')
REPORT_DIR = ARTIFACT_ROOT / 'reports'
PLOT_DIR = ARTIFACT_ROOT / 'plots'
DASHBOARD_DIR = ARTIFACT_ROOT / 'dashboard'
SFT_ARTIFACT_DIR = Path('artifacts/trl-openenv-hf-a10g-qwen-rich')
SELFPLAY_DIR = ARTIFACT_ROOT / 'selfplay-0.5b'
GRPO_DIR = ARTIFACT_ROOT / 'grpo-0.5b'
SFT15_DIR = ARTIFACT_ROOT / 'sft-1.5b'
DPO_DIR = ARTIFACT_ROOT / 'dpo-falsifier-distill'

print(json.dumps({
    'run_profile': RUN_PROFILE,
    'run_dpo': RUN_DPO,
    'artifact_root': str(ARTIFACT_ROOT),
    'description': cfg['description'],
}, indent=2))


In [ ]:
def cli_args(flag_map):
    args = []
    for key, value in flag_map.items():
        flag = f"--{key.replace('_', '-')}"
        if isinstance(value, bool):
            if value:
                args.append(flag)
        elif isinstance(value, (list, tuple)):
            args.append(flag)
            args.extend(str(item) for item in value)
        elif value is not None:
            args.extend([flag, str(value)])
    return args

def run_cmd(cmd):
    pretty = ' '.join(shlex.quote(str(part)) for part in cmd)
    print(f"\n$ {pretty}\n")
    subprocess.run([str(part) for part in cmd], check=True)

REPORT_DIR.mkdir(parents=True, exist_ok=True)
excluded = ['grpo-1.5b', 'grpo-3b-flagship']
if not RUN_DPO:
    excluded.append('dpo-falsifier-distill')
launch_ledger = {
    'jobs': [
        {'name': name, 'exclude_from_live_reports': True}
        for name in excluded
    ]
}
(REPORT_DIR / 'hf_exquisite_launches.json').write_text(json.dumps(launch_ledger, indent=2), encoding='utf-8')
print('Initialized report exclusions for notebook-only run scope')


## 1. Self-Play Candidate Generation

In [ ]:
run_cmd([
    'python', 'training/exquisite/collect_selfplay_rollouts.py',
    '--output-dir', SELFPLAY_DIR,
    '--sft-artifact-dir', SFT_ARTIFACT_DIR,
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
] + cli_args(cfg['selfplay']))


## 2. GRPO Environment-Reward Training

In [ ]:
run_cmd([
    'python', 'training/exquisite/grpo_env_reward_training.py',
    '--output-dir', GRPO_DIR,
    '--sft-artifact-dir', SFT_ARTIFACT_DIR,
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
] + cli_args(cfg['grpo']))


## 3. Fast-Profile 1.5B SFT Scaling Run

In [ ]:
run_cmd([
    'python', 'training/ledgershield_trl_training.py',
    '--output-dir', SFT15_DIR,
    '--case-split', 'all',
    '--train',
    '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '4',
    '--max-seq-length', '4096',
] + cli_args(cfg['sft_1_5b']))


## 4. Optional DPO Falsifier Distillation

In [ ]:
if RUN_DPO:
    run_cmd([
        'python', 'training/exquisite/dpo_falsifier_distill.py',
        '--output-dir', DPO_DIR,
        '--preference-file', SELFPLAY_DIR / 'falsifier_preferences.jsonl',
        '--sft-artifact-dir', SFT15_DIR,
        '--adapter-path', SFT15_DIR / 'final_model',
        '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    ] + cli_args(cfg['dpo']))
else:
    print('RUN_DPO is False, skipping DPO distillation')


## 5. Rebuild The Final Matrix, Plots, Dashboard, And Report

In [ ]:
run_cmd([
    'python', 'training/exquisite/evaluate_exquisite_policy.py',
    '--artifact-root', ARTIFACT_ROOT,
    '--output-dir', REPORT_DIR,
])

run_cmd([
    'python', 'training/exquisite/plot_exquisite_training_results.py',
    '--artifact-root', ARTIFACT_ROOT,
    '--report-dir', REPORT_DIR,
    '--output-dir', PLOT_DIR,
])

run_cmd([
    'python', 'training/exquisite/build_exquisite_dashboard.py',
    '--artifact-root', ARTIFACT_ROOT,
    '--report-dir', REPORT_DIR,
    '--plot-dir', PLOT_DIR,
    '--output-dir', DASHBOARD_DIR,
])

run_cmd([
    'python', 'training/exquisite/render_exquisite_report.py',
    '--artifact-root', ARTIFACT_ROOT,
    '--report-dir', REPORT_DIR,
    '--dashboard-dir', DASHBOARD_DIR,
])


## 6. Inspect Outputs

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

summary = json.loads((REPORT_DIR / 'exquisite_training_summary.json').read_text())
print(json.dumps(summary, indent=2))

print('\nFinal policy matrix:\n')
print((REPORT_DIR / 'final_policy_matrix.csv').read_text())

for plot_name in [
    '01_final_policy_ladder.png',
    '08_grpo_reward_curve_smoothed.png',
    '04_score_safety_frontier_all_policies.png',
    '27_per_case_score_heatmap.png',
]:
    plot_path = PLOT_DIR / plot_name
    if plot_path.exists():
        print(plot_path)
        display(Image(filename=str(plot_path)))

print('\nDashboard:', DASHBOARD_DIR / 'index.html')
print('Report:', REPORT_DIR / 'exquisite_training_report.md')
